In [ ]:
!cp "/content/drive/MyDrive/Hate Speech LLMs/ARR Data/LLM Evaluation/qwenThinkData.xlsx" .

In [ ]:
!pip install bert_score sacrebleu rouge-score sentence-transformers

In [ ]:
!pip install -q "transformers<5" "tokenizers<0.22" "bert-score==0.3.12"

In [ ]:
import numpy as np
import pandas as pd
from bert_score import score as bert_score
from sacrebleu.metrics import CHRF
from rouge_score import rouge_scorer
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
df = pd.read_excel("qwenThinkData.xlsx")

In [ ]:
ref_col = 'Hop_6_Narrative (Ref)'
pred_col = 'Hop_6_Narrative'

In [ ]:
df = df[df["Hop_1_Triage (Ref)"] != "بی‌ضرر"]
df[ref_col] = df[ref_col].fillna('')
df[pred_col] = df[pred_col].fillna('')

## BERTScore-F1

In [ ]:
P, R, F1 = bert_score(
    df[pred_col].tolist(),
    df[ref_col].tolist(),
    model_type="HooshvareLab/bert-base-parsbert-uncased",
    num_layers=12,
    lang="fa",
    verbose=True
)

bert_score_f1 = F1.mean().item()

print(f"BERTScore-F1: {bert_score_f1:.4f}")

calculating scores...
computing bert embedding.


  0%|          | 0/27 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/15 [00:00<?, ?it/s]

done in 157.16 seconds, 5.73 sentences/sec
BERTScore-F1: 0.4739


## chrF++

In [ ]:
chrf = CHRF(word_order=2)

chrf_scores = []

for ref, pred in zip(df[ref_col], df[pred_col]):
    score = chrf.sentence_score(pred, [ref])
    chrf_scores.append(score.score)

chrf_score = np.mean(chrf_scores)

print(f"chrF++: {chrf_score:.4f}")

chrF++: 19.4670


## ROUGE-L

In [ ]:
scorer = rouge_scorer.RougeScorer(
    ['rougeL'],
    use_stemmer=False
)

rouge_scores = []

for ref, pred in zip(df[ref_col], df[pred_col]):
    score = scorer.score(ref, pred)
    rouge_scores.append(score['rougeL'].fmeasure)

rouge_l = np.mean(rouge_scores)

print(f"ROUGE-L: {rouge_l:.4f}")

ROUGE-L: 0.0000


## Cosine similarity

In [ ]:
model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
)

ref_embeddings = model.encode(
    df[ref_col].tolist(),
    convert_to_numpy=True,
    show_progress_bar=True
)

pred_embeddings = model.encode(
    df[pred_col].tolist(),
    convert_to_numpy=True,
    show_progress_bar=True
)

cosine_scores = []

for ref_embedding, pred_embedding in zip(ref_embeddings, pred_embeddings):
    score = cosine_similarity(
        [ref_embedding],
        [pred_embedding]
    )[0][0]

    cosine_scores.append(score)

cosine_similarity_score = np.mean(cosine_scores)

print(f"Cosine Similarity: {cosine_similarity_score:.4f}")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/723 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/29 [00:00<?, ?it/s]

Batches:   0%|          | 0/29 [00:00<?, ?it/s]

Cosine Similarity: 0.6033
